In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict
from itertools import combinations

In [2]:
#SETTING UP DATAFRAMES

currdf = pd.read_csv('cbb25.csv')
pastdf = pd.read_csv('pastCBB.csv')

pastdf['Win%'] = pastdf['W'] / pastdf['G']
currdf['Win%'] = currdf['W'] / currdf['G']


In [3]:
#Thresholds for each round

important_stats = ['ADJOE', 'ADJDE', 'BARTHAG', 'EFG_O', 'EFG_D', 'TOR', 'ORB', '3P%', 'ADJ_T', 'Win%']

# Group by Result (1,2,4,8,16,32) and calculate mean
averages_by_round = pastdf.groupby('Result')[important_stats].mean()

# Show the result
print(averages_by_round)

             ADJOE      ADJDE   BARTHAG      EFG_O      EFG_D        TOR  \
Result                                                                     
1.0     122.286687  91.272563  0.963402  54.058638  45.575988  16.771550   
2.0     118.096688  91.791875  0.940541  53.103262  46.325756  16.709431   
4.0     117.114712  93.463125  0.921795  52.441441  46.594900  17.416697   
8.0     117.177411  93.600284  0.921039  53.164681  46.694725  17.283939   
16.0    115.409891  94.044330  0.903231  52.562048  46.839124  17.414558   
32.0    113.064102  95.808134  0.855569  51.895441  47.442521  17.676014   
64.0    109.002855  99.247947  0.714916  51.526557  48.094123  18.268310   

              ORB        3P%      ADJ_T      Win%  
Result                                             
1.0     35.858694  37.461250  66.389656  0.876192  
2.0     33.694406  35.742638  66.406419  0.837372  
4.0     34.322875  36.284534  66.022831  0.789856  
8.0     33.624053  36.021558  66.715567  0.782193  
16.

In [4]:
# Offense rating categories (ADJOE)
pastdf['Offense_Category'] = 'Average_Offense'
pastdf.loc[pastdf['ADJOE'] <= 110, 'Offense_Category'] = 'Very_Bad_Offense'
pastdf.loc[(pastdf['ADJOE'] > 110) & (pastdf['ADJOE'] <= 114), 'Offense_Category'] = 'Bad_Offense'
pastdf.loc[(pastdf['ADJOE'] > 114) & (pastdf['ADJOE'] <= 117), 'Offense_Category'] = 'Average_Offense'
pastdf.loc[(pastdf['ADJOE'] > 117) & (pastdf['ADJOE'] <= 120), 'Offense_Category'] = 'Good_Offense'
pastdf.loc[pastdf['ADJOE'] > 120, 'Offense_Category'] = 'Very_Good_Offense'

# Defense rating categories (ADJDE) - lower is better
pastdf['Defense_Category'] = 'Average_Defense'
pastdf.loc[pastdf['ADJDE'] <= 92, 'Defense_Category'] = 'Very_Good_Defense'
pastdf.loc[(pastdf['ADJDE'] > 92) & (pastdf['ADJDE'] <= 94), 'Defense_Category'] = 'Good_Defense'
pastdf.loc[(pastdf['ADJDE'] > 94) & (pastdf['ADJDE'] <= 96), 'Defense_Category'] = 'Average_Defense'
pastdf.loc[(pastdf['ADJDE'] > 96) & (pastdf['ADJDE'] <= 98), 'Defense_Category'] = 'Bad_Defense'
pastdf.loc[pastdf['ADJDE'] > 98, 'Defense_Category'] = 'Very_Bad_Defense'

# Overall strength (BARTHAG)
pastdf['Team_Strength_Category'] = 'Average_Team'
pastdf.loc[pastdf['BARTHAG'] <= 0.75, 'Team_Strength_Category'] = 'Very_Bad_Team'
pastdf.loc[(pastdf['BARTHAG'] > 0.75) & (pastdf['BARTHAG'] <= 0.85), 'Team_Strength_Category'] = 'Bad_Team'
pastdf.loc[(pastdf['BARTHAG'] > 0.85) & (pastdf['BARTHAG'] <= 0.93), 'Team_Strength_Category'] = 'Average_Team'
pastdf.loc[(pastdf['BARTHAG'] > 0.93) & (pastdf['BARTHAG'] <= 0.95), 'Team_Strength_Category'] = 'Good_Team'
pastdf.loc[pastdf['BARTHAG'] > 0.95, 'Team_Strength_Category'] = 'Very_Good_Team'

# Effective FG% offense (EFG_O)
pastdf['Shooting_Efficiency_Category'] = 'Average_EFG'
pastdf.loc[pastdf['EFG_O'] <= 50, 'Shooting_Efficiency_Category'] = 'Very_Bad_EFG'
pastdf.loc[(pastdf['EFG_O'] > 50) & (pastdf['EFG_O'] <= 51.5), 'Shooting_Efficiency_Category'] = 'Bad_EFG'
pastdf.loc[(pastdf['EFG_O'] > 51.5) & (pastdf['EFG_O'] <= 52.5), 'Shooting_Efficiency_Category'] = 'Average_EFG'
pastdf.loc[(pastdf['EFG_O'] > 52.5) & (pastdf['EFG_O'] <= 54), 'Shooting_Efficiency_Category'] = 'Good_EFG'
pastdf.loc[pastdf['EFG_O'] > 54, 'Shooting_Efficiency_Category'] = 'Very_Good_EFG'

# Effective FG% defense (EFG_D) - lower is better
pastdf['Defense_Efficiency_Category'] = 'Average_EFG_D'
pastdf.loc[pastdf['EFG_D'] <= 45, 'Defense_Efficiency_Category'] = 'Very_Good_EFG_D'
pastdf.loc[(pastdf['EFG_D'] > 45) & (pastdf['EFG_D'] <= 46.5), 'Defense_Efficiency_Category'] = 'Good_EFG_D'
pastdf.loc[(pastdf['EFG_D'] > 46.5) & (pastdf['EFG_D'] <= 48), 'Defense_Efficiency_Category'] = 'Average_EFG_D'
pastdf.loc[(pastdf['EFG_D'] > 48) & (pastdf['EFG_D'] <= 49), 'Defense_Efficiency_Category'] = 'Bad_EFG_D'
pastdf.loc[pastdf['EFG_D'] > 49, 'Defense_Efficiency_Category'] = 'Very_Bad_EFG_D'

# Turnover Rate (TOR) - lower is better
pastdf['Turnover_Category'] = 'Average_TOR'
pastdf.loc[pastdf['TOR'] <= 16, 'Turnover_Category'] = 'Very_Good_TOR'
pastdf.loc[(pastdf['TOR'] > 16) & (pastdf['TOR'] <= 17.2), 'Turnover_Category'] = 'Good_TOR'
pastdf.loc[(pastdf['TOR'] > 17.2) & (pastdf['TOR'] <= 18.5), 'Turnover_Category'] = 'Average_TOR'
pastdf.loc[(pastdf['TOR'] > 18.5) & (pastdf['TOR'] <= 19.5), 'Turnover_Category'] = 'Bad_TOR'
pastdf.loc[pastdf['TOR'] > 19.5, 'Turnover_Category'] = 'Very_Bad_TOR'

# Offensive Rebounding (ORB)
pastdf['Rebounding_Category'] = 'Average_ORB'
pastdf.loc[pastdf['ORB'] <= 30, 'Rebounding_Category'] = 'Very_Bad_ORB'
pastdf.loc[(pastdf['ORB'] > 30) & (pastdf['ORB'] <= 32), 'Rebounding_Category'] = 'Bad_ORB'
pastdf.loc[(pastdf['ORB'] > 32) & (pastdf['ORB'] <= 33), 'Rebounding_Category'] = 'Average_ORB'
pastdf.loc[(pastdf['ORB'] > 33) & (pastdf['ORB'] <= 35), 'Rebounding_Category'] = 'Good_ORB'
pastdf.loc[pastdf['ORB'] > 35, 'Rebounding_Category'] = 'Very_Good_ORB'

# Three-Point Percentage (3P%)
pastdf['ThreePoint_Category'] = 'Average_3P'
pastdf.loc[pastdf['3P%'] <= 33, 'ThreePoint_Category'] = 'Very_Bad_3P'
pastdf.loc[(pastdf['3P%'] > 33) & (pastdf['3P%'] <= 35), 'ThreePoint_Category'] = 'Bad_3P'
pastdf.loc[(pastdf['3P%'] > 35) & (pastdf['3P%'] <= 36.5), 'ThreePoint_Category'] = 'Average_3P'
pastdf.loc[(pastdf['3P%'] > 36.5) & (pastdf['3P%'] <= 38), 'ThreePoint_Category'] = 'Good_3P'
pastdf.loc[pastdf['3P%'] > 38, 'ThreePoint_Category'] = 'Very_Good_3P'

# Win Percentage (Win%)
pastdf['WinRate_Category'] = 'Average_Win%'
pastdf.loc[pastdf['Win%'] <= 0.68, 'WinRate_Category'] = 'Very_Bad_Win%'
pastdf.loc[(pastdf['Win%'] > 0.68) & (pastdf['Win%'] <= 0.75), 'WinRate_Category'] = 'Bad_Win%'
pastdf.loc[(pastdf['Win%'] > 0.75) & (pastdf['Win%'] <= 0.78), 'WinRate_Category'] = 'Average_Win%'
pastdf.loc[(pastdf['Win%'] > 0.78) & (pastdf['Win%'] <= 0.85), 'WinRate_Category'] = 'Good_Win%'
pastdf.loc[pastdf['Win%'] > 0.85, 'WinRate_Category'] = 'Very_Good_Win%'


# Re-add round-based success labels
pastdf['IsChampion'] = pastdf['Result'] == 1
pastdf['MadeFinals'] = pastdf['Result'] <= 2
pastdf['MadeFinal4'] = pastdf['Result'] <= 4
pastdf['MadeElite8'] = pastdf['Result'] <= 8
pastdf['MadeSweet16'] = pastdf['Result'] <= 16
pastdf['MadeRound32'] = pastdf['Result'] <= 32

only64 = pastdf[pastdf['Result'] == 64]
only32 = pastdf[pastdf['Result'] == 32]
only16 = pastdf[pastdf['Result'] == 16]
only8 = pastdf[pastdf['Result'] == 8]
only4 = pastdf[pastdf['Result'] == 4]
only2 = pastdf[pastdf['Result'] == 2]

made32 = pastdf[pastdf['Result'] <=32]
made16 = pastdf[pastdf['Result'] <=16]
made8 = pastdf[pastdf['Result'] <=8]
made4 = pastdf[pastdf['Result'] <=4]
made2 = pastdf[pastdf['Result'] <=2]
made1 = pastdf[pastdf['Result'] <=1]

In [5]:
category_columns = [
    'Offense_Category', 'Defense_Category', 'Team_Strength_Category',
    'Shooting_Efficiency_Category', 'Defense_Efficiency_Category',
    'Turnover_Category', 'Rebounding_Category', 'ThreePoint_Category', 'WinRate_Category'
]

transactions_made_32 = []
transactions_made_16 = []
transactions_made_8 = []
transactions_made_4 = []
transactions_made_2 = []
transactions1 = []

transactions_only_64 = []
transactions_only_32 = []
transactions_only_16 = []
transactions_only_8 = []
transactions_only_4 = []
transactions_only_2 = []

# For teams that made Round of 32
for index, row in made32.iterrows():
    transaction = set(row[col] for col in category_columns)
    transactions_made_32.append(transaction)

# For teams that made Sweet 16
for index, row in made16.iterrows():
    transaction = set(row[col] for col in category_columns)
    transactions_made_16.append(transaction)

# For teams that made Elite 8
for index, row in made8.iterrows():
    transaction = set(row[col] for col in category_columns)
    transactions_made_8.append(transaction)

# For teams that made Final 4
for index, row in made4.iterrows():
    transaction = set(row[col] for col in category_columns)
    transactions_made_4.append(transaction)

# For teams that made Finals
for index, row in made2.iterrows():
    transaction = set(row[col] for col in category_columns)
    transactions_made_2.append(transaction)

# For teams that won the Championship
for index, row in made1.iterrows():
    transaction = set(row[col] for col in category_columns)
    transactions1.append(transaction)

for index, row in only64.iterrows():
    transaction = set(row[col] for col in category_columns)
    transactions_only_64.append(transaction)

for index, row in only32.iterrows():
    transaction = set(row[col] for col in category_columns)
    transactions_only_32.append(transaction)

for index, row in only16.iterrows():
    transaction = set(row[col] for col in category_columns)
    transactions_only_16.append(transaction)

for index, row in only8.iterrows():
    transaction = set(row[col] for col in category_columns)
    transactions_only_8.append(transaction)

for index, row in only4.iterrows():
    transaction = set(row[col] for col in category_columns)
    transactions_only_4.append(transaction)

for index, row in only2.iterrows():
    transaction = set(row[col] for col in category_columns)
    transactions_only_2.append(transaction)


In [6]:
def find_frequent_1_itemsets(transactions, min_support):
    item_counts = defaultdict(int)
    total_transactions = len(transactions)
    
    for transaction in transactions:
        for item in transaction:
            item_counts[item] += 1
    
    frequent_items = {}
    for item, count in item_counts.items():
        support = count / total_transactions
        if support >= min_support:
            frequent_items[frozenset([item])] = support
    
    return frequent_items

def generate_candidates(frequent_itemsets, k):
    """
    Generate candidate itemsets of size k from frequent itemsets of size k-1
    """
    items = list(frequent_itemsets.keys())
    candidates = set()
    
    for i in range(len(items)):
        for j in range(i+1, len(items)):
            candidate = items[i] | items[j]
            if len(candidate) == k:
                candidates.add(candidate)
    
    return candidates

def count_support(candidates, transactions, min_support):
    """
    Count support for each candidate itemset and filter by min_support
    """
    candidate_counts = defaultdict(int)
    total_transactions = len(transactions)
    
    for transaction in transactions:
        for candidate in candidates:
            if candidate.issubset(transaction):
                candidate_counts[candidate] += 1
                
    # Filter by support
    frequent_itemsets = {}
    for candidate, count in candidate_counts.items():
        support = count / total_transactions
        if support >= min_support:
            frequent_itemsets[candidate] = support
    
    return frequent_itemsets

In [7]:
# Offense rating categories (ADJOE)
currdf['Offense_Category'] = 'Average_Offense'
currdf.loc[currdf['ADJOE'] <= 110, 'Offense_Category'] = 'Very_Bad_Offense'
currdf.loc[(currdf['ADJOE'] > 110) & (currdf['ADJOE'] <= 114), 'Offense_Category'] = 'Bad_Offense'
currdf.loc[(currdf['ADJOE'] > 114) & (currdf['ADJOE'] <= 117), 'Offense_Category'] = 'Average_Offense'
currdf.loc[(currdf['ADJOE'] > 117) & (currdf['ADJOE'] <= 120), 'Offense_Category'] = 'Good_Offense'
currdf.loc[currdf['ADJOE'] > 120, 'Offense_Category'] = 'Very_Good_Offense'

# Defense rating categories (ADJDE) - lower is better
currdf['Defense_Category'] = 'Average_Defense'
currdf.loc[currdf['ADJDE'] <= 92, 'Defense_Category'] = 'Very_Good_Defense'
currdf.loc[(currdf['ADJDE'] > 92) & (currdf['ADJDE'] <= 94), 'Defense_Category'] = 'Good_Defense'
currdf.loc[(currdf['ADJDE'] > 94) & (currdf['ADJDE'] <= 96), 'Defense_Category'] = 'Average_Defense'
currdf.loc[(currdf['ADJDE'] > 96) & (currdf['ADJDE'] <= 98), 'Defense_Category'] = 'Bad_Defense'
currdf.loc[currdf['ADJDE'] > 98, 'Defense_Category'] = 'Very_Bad_Defense'

# Overall strength (BARTHAG)
currdf['Team_Strength_Category'] = 'Average_Team'
currdf.loc[currdf['BARTHAG'] <= 0.75, 'Team_Strength_Category'] = 'Very_Bad_Team'
currdf.loc[(currdf['BARTHAG'] > 0.75) & (currdf['BARTHAG'] <= 0.85), 'Team_Strength_Category'] = 'Bad_Team'
currdf.loc[(currdf['BARTHAG'] > 0.85) & (currdf['BARTHAG'] <= 0.93), 'Team_Strength_Category'] = 'Average_Team'
currdf.loc[(currdf['BARTHAG'] > 0.93) & (currdf['BARTHAG'] <= 0.95), 'Team_Strength_Category'] = 'Good_Team'
currdf.loc[currdf['BARTHAG'] > 0.95, 'Team_Strength_Category'] = 'Very_Good_Team'

# Effective FG% offense (EFG_O)
currdf['Shooting_Efficiency_Category'] = 'Average_EFG'
currdf.loc[currdf['EFG_O'] <= 50, 'Shooting_Efficiency_Category'] = 'Very_Bad_EFG'
currdf.loc[(currdf['EFG_O'] > 50) & (currdf['EFG_O'] <= 51.5), 'Shooting_Efficiency_Category'] = 'Bad_EFG'
currdf.loc[(currdf['EFG_O'] > 51.5) & (currdf['EFG_O'] <= 52.5), 'Shooting_Efficiency_Category'] = 'Average_EFG'
currdf.loc[(currdf['EFG_O'] > 52.5) & (currdf['EFG_O'] <= 54), 'Shooting_Efficiency_Category'] = 'Good_EFG'
currdf.loc[currdf['EFG_O'] > 54, 'Shooting_Efficiency_Category'] = 'Very_Good_EFG'

# Effective FG% defense (EFG_D) - lower is better
currdf['Defense_Efficiency_Category'] = 'Average_EFG_D'
currdf.loc[currdf['EFG_D'] <= 45, 'Defense_Efficiency_Category'] = 'Very_Good_EFG_D'
currdf.loc[(currdf['EFG_D'] > 45) & (currdf['EFG_D'] <= 46.5), 'Defense_Efficiency_Category'] = 'Good_EFG_D'
currdf.loc[(currdf['EFG_D'] > 46.5) & (currdf['EFG_D'] <= 48), 'Defense_Efficiency_Category'] = 'Average_EFG_D'
currdf.loc[(currdf['EFG_D'] > 48) & (currdf['EFG_D'] <= 49), 'Defense_Efficiency_Category'] = 'Bad_EFG_D'
currdf.loc[currdf['EFG_D'] > 49, 'Defense_Efficiency_Category'] = 'Very_Bad_EFG_D'

# Turnover Rate (TOR) - lower is better
currdf['Turnover_Category'] = 'Average_TOR'
currdf.loc[currdf['TOR'] <= 16, 'Turnover_Category'] = 'Very_Good_TOR'
currdf.loc[(currdf['TOR'] > 16) & (currdf['TOR'] <= 17.2), 'Turnover_Category'] = 'Good_TOR'
currdf.loc[(currdf['TOR'] > 17.2) & (currdf['TOR'] <= 18.5), 'Turnover_Category'] = 'Average_TOR'
currdf.loc[(currdf['TOR'] > 18.5) & (currdf['TOR'] <= 19.5), 'Turnover_Category'] = 'Bad_TOR'
currdf.loc[currdf['TOR'] > 19.5, 'Turnover_Category'] = 'Very_Bad_TOR'

# Offensive Rebounding (ORB)
currdf['Rebounding_Category'] = 'Average_ORB'
currdf.loc[currdf['ORB'] <= 30, 'Rebounding_Category'] = 'Very_Bad_ORB'
currdf.loc[(currdf['ORB'] > 30) & (currdf['ORB'] <= 32), 'Rebounding_Category'] = 'Bad_ORB'
currdf.loc[(currdf['ORB'] > 32) & (currdf['ORB'] <= 33), 'Rebounding_Category'] = 'Average_ORB'
currdf.loc[(currdf['ORB'] > 33) & (currdf['ORB'] <= 35), 'Rebounding_Category'] = 'Good_ORB'
currdf.loc[currdf['ORB'] > 35, 'Rebounding_Category'] = 'Very_Good_ORB'

# Three-Point Percentage (3P%)
currdf['ThreePoint_Category'] = 'Average_3P'
currdf.loc[currdf['3P%'] <= 33, 'ThreePoint_Category'] = 'Very_Bad_3P'
currdf.loc[(currdf['3P%'] > 33) & (currdf['3P%'] <= 35), 'ThreePoint_Category'] = 'Bad_3P'
currdf.loc[(currdf['3P%'] > 35) & (currdf['3P%'] <= 36.5), 'ThreePoint_Category'] = 'Average_3P'
currdf.loc[(currdf['3P%'] > 36.5) & (currdf['3P%'] <= 38), 'ThreePoint_Category'] = 'Good_3P'
currdf.loc[currdf['3P%'] > 38, 'ThreePoint_Category'] = 'Very_Good_3P'

# Win Percentage (Win%)
currdf['WinRate_Category'] = 'Average_Win%'
currdf.loc[currdf['Win%'] <= 0.68, 'WinRate_Category'] = 'Very_Bad_Win%'
currdf.loc[(currdf['Win%'] > 0.68) & (currdf['Win%'] <= 0.75), 'WinRate_Category'] = 'Bad_Win%'
currdf.loc[(currdf['Win%'] > 0.75) & (currdf['Win%'] <= 0.78), 'WinRate_Category'] = 'Average_Win%'
currdf.loc[(currdf['Win%'] > 0.78) & (currdf['Win%'] <= 0.85), 'WinRate_Category'] = 'Good_Win%'
currdf.loc[currdf['Win%'] > 0.85, 'WinRate_Category'] = 'Very_Good_Win%'

**ROUND OF 64 ITEMSETS**

In [8]:
min_support = .2
frequent_1_itemsets64 = find_frequent_1_itemsets(transactions_only_64, min_support)

for itemset, support in frequent_1_itemsets64.items():
    print(itemset, support)

frozenset({'Average_TOR'}) 0.25573192239858905
frozenset({'Bad_Offense'}) 0.23985890652557318
frozenset({'Bad_Win%'}) 0.26278659611992944
frozenset({'Average_Team'}) 0.26102292768959434
frozenset({'Very_Bad_Win%'}) 0.5185185185185185
frozenset({'Bad_3P'}) 0.2751322751322751
frozenset({'Very_Bad_TOR'}) 0.2768959435626102
frozenset({'Very_Bad_EFG'}) 0.30335097001763667
frozenset({'Very_Bad_EFG_D'}) 0.3544973544973545
frozenset({'Very_Bad_Defense'}) 0.5679012345679012
frozenset({'Average_EFG_D'}) 0.2345679012345679
frozenset({'Bad_Team'}) 0.2786596119929453
frozenset({'Very_Bad_Offense'}) 0.5573192239858906
frozenset({'Very_Bad_ORB'}) 0.3527336860670194
frozenset({'Very_Bad_Team'}) 0.43915343915343913


In [9]:
candidates64_2 = generate_candidates(frequent_1_itemsets64, k=2)

frequent_2_itemsets64 = count_support(candidates64_2, transactions_only_64, min_support=0.3)

for itemset, support in frequent_2_itemsets64.items():
    print(itemset, support)

frozenset({'Very_Bad_Defense', 'Very_Bad_Win%'}) 0.3333333333333333
frozenset({'Very_Bad_Defense', 'Very_Bad_EFG_D'}) 0.31040564373897706
frozenset({'Very_Bad_Offense', 'Very_Bad_Win%'}) 0.31569664902998235
frozenset({'Very_Bad_Defense', 'Very_Bad_Team'}) 0.3932980599647266
frozenset({'Very_Bad_Offense', 'Very_Bad_Team'}) 0.37742504409171074
frozenset({'Very_Bad_Defense', 'Very_Bad_Offense'}) 0.3403880070546737


INTERPRETATION: 

About 40% of the teams that got eliminated in the first round had a very bad defensive rating and a very bad team rating. 

About 38% of the teams that got eliminated in the first round had a very bad offensive rating and a very bad team rating. 

About 34% of the teams that got eliminated in the first round had a very bad offensive rating and a very defensive rating rating. 

About 34% of the teams that got eliminated in the first round had a very bad defensive rating and a very bad win%. 

About 31% of the teams that got eliminated in the first round had a very bad offensive rating and a very bad win%. 

About 31% of the teams that got eliminated in the first round had a very bad defensive rating and a very bad EFGD rating. 


In [10]:
bad_def_and_team = currdf[
    (currdf['Defense_Category'] == 'Very_Bad_Defense') &
    (currdf['Team_Strength_Category'] == 'Very_Bad_Team')
]

bad_off_and_team = currdf[
    (currdf['Offense_Category'] == 'Very_Bad_Offense') &
    (currdf['Team_Strength_Category'] == 'Very_Bad_Team')
]

bad_off_and_def = currdf[
    (currdf['Offense_Category'] == 'Very_Bad_Offense') &
    (currdf['Defense_Category'] == 'Very_Bad_Defense')
]

bad_win_and_def = currdf[
    (currdf['WinRate_Category'] == 'Very_Bad_Win%') &
    (currdf['Defense_Category'] == 'Very_Bad_Defense')
]

bad_win_and_off = currdf[
    (currdf['WinRate_Category'] == 'Very_Bad_Win%') &
    (currdf['Offense_Category'] == 'Very_Bad_Offense')
]

bad_efgd_and_def = currdf[
    (currdf['Defense_Efficiency_Category'] == 'Very_Bad_EFG_D') &
    (currdf['Defense_Category'] == 'Very_Bad_Defense')
]

teams_bad_def_and_team = set(bad_def_and_team['Team'])
teams_bad_off_and_team = set(bad_off_and_team['Team'])
teams_bad_off_and_def = set(bad_off_and_def['Team'])
teams_bad_win_and_def = set(bad_win_and_def['Team'])
teams_bad_win_and_off = set(bad_win_and_off['Team'])
teams_bad_efgd_and_def = set(bad_efgd_and_def['Team'])

bad_combo_count = defaultdict(int)

Add +1 for every bad trait set a team appears in
for team in teams_bad_def_and_team:
    bad_combo_count[team] += 1
for team in teams_bad_off_and_team:
    bad_combo_count[team] += 1
for team in teams_bad_off_and_def:
    bad_combo_count[team] += 1
for team in teams_bad_win_and_def:
    bad_combo_count[team] += 1
for team in teams_bad_win_and_off:
    bad_combo_count[team] += 1
for team in teams_bad_efgd_and_def:
    bad_combo_count[team] += 1

Print teams sorted by how many bad combos they match
sorted_bad_teams = sorted(bad_combo_count.items(), key=lambda x: x[1], reverse=True)

print("Teams ranked by number of very bad frequent combinations:")
for team, count in sorted_bad_teams:
    print(f"{team}: {count} bad frequent combinations")
    
print(len(sorted_bad_teams))

Teams ranked by number of very bad frequent combinations:
Saint_Francis: 6 bad frequent combinations
American: 6 bad frequent combinations
Alabama_St_: 6 bad frequent combinations
Mount_St__Mary_s: 5 bad frequent combinations
SIU_Edwardsville: 5 bad frequent combinations
Bryant: 5 bad frequent combinations
Norfolk_St_: 4 bad frequent combinations
Robert_Morris: 4 bad frequent combinations
Grand_Canyon: 3 bad frequent combinations
Wofford: 3 bad frequent combinations
Troy: 3 bad frequent combinations
Nebraska_Omaha: 3 bad frequent combinations
Montana: 2 bad frequent combinations
High_Point: 2 bad frequent combinations
UNC_Wilmington: 2 bad frequent combinations
Akron: 2 bad frequent combinations
Vanderbilt: 2 bad frequent combinations
Baylor: 2 bad frequent combinations
North_Carolina: 2 bad frequent combinations
Purdue: 2 bad frequent combinations
Mississippi_St_: 2 bad frequent combinations
Oklahoma: 2 bad frequent combinations
Missouri: 2 bad frequent combinations
Kentucky: 2 bad fr

These are the teams that are most likely going to be eliminated in the first round. If both teams in a matchup are lsited here, we can compare how many bad combinations each team has and pick the team with the lesser amount. 

**Round of 32 itemsets**

In [11]:
min_support = .2
frequent_1_itemsets32m = find_frequent_1_itemsets(transactions_made_32, min_support)

candidates32m_2 = generate_candidates(frequent_1_itemsets32m, k=2)

frequent_2_itemsets32m = count_support(candidates32m_2, transactions_made_32, min_support=0.15)

for itemset, support in frequent_2_itemsets32m.items():
    print(itemset, support)

frozenset({'Very_Good_ORB', 'Average_Team'}) 0.162109375
frozenset({'Bad_Offense', 'Average_Team'}) 0.1796875
frozenset({'Very_Bad_Win%', 'Average_Team'}) 0.15234375
frozenset({'Bad_Win%', 'Average_Team'}) 0.169921875


In [12]:
avg_team_and_very_good_ORB = currdf[
    (currdf['Rebounding_Category'] == 'Very_Good_ORB') &
    (currdf['Team_Strength_Category'] == 'Average_Team')
]

avg_team_and_bad_off = currdf[
    (currdf['Offense_Category'] == 'Bad_Offense') &
    (currdf['Team_Strength_Category'] == 'Average_Team')
]

avg_team_and_vbad_win = currdf[
    (currdf['WinRate_Category'] == 'Very_Bad_Win%') &
    (currdf['Team_Strength_Category'] == 'Average_Team')
]

avg_team_and_bad_win = currdf[
    (currdf['WinRate_Category'] == 'Bad_Win%') &
    (currdf['Team_Strength_Category'] == 'Average_Team')
]

teams_avg_team_and_very_good_ORB = set(avg_team_and_very_good_ORB['Team'])
teams_avg_team_and_bad_off = set(avg_team_and_bad_off['Team'])
teams_avg_team_and_vbad_win = set(avg_team_and_vbad_win['Team'])
teams_avg_team_and_bad_win = set(avg_team_and_bad_win['Team'])

combo_count32 = defaultdict(int)

# Add +1 for every bad trait set a team appears in
for team in teams_avg_team_and_very_good_ORB:
    combo_count32[team] += 1
for team in teams_avg_team_and_bad_off:
    combo_count32[team] += 1
for team in teams_avg_team_and_vbad_win:
    combo_count32[team] += 1
for team in teams_avg_team_and_bad_win:
    combo_count32[team] += 1

# Print teams sorted by how many combos they match
sorted_teams32 = sorted(combo_count32.items(), key=lambda x: x[1], reverse=True)

print("Teams ranked by number frequent combinations:")
for team, count in sorted_teams32:
    print(f"{team}: {count} frequent combinations")
    
print(len(sorted_teams32))

Teams ranked by number frequent combinations:
Baylor: 2 frequent combinations
Georgia: 2 frequent combinations
Illinois: 2 frequent combinations
Texas_A_M: 2 frequent combinations
Connecticut: 2 frequent combinations
Arkansas: 2 frequent combinations
Saint_Mary_s: 1 frequent combinations
VCU: 1 frequent combinations
UC_San_Diego: 1 frequent combinations
New_Mexico: 1 frequent combinations
Kansas: 1 frequent combinations
Xavier: 1 frequent combinations
Vanderbilt: 1 frequent combinations
Mississippi: 1 frequent combinations
North_Carolina: 1 frequent combinations
Purdue: 1 frequent combinations
Texas: 1 frequent combinations
Oklahoma: 1 frequent combinations
Mississippi_St_: 1 frequent combinations
Marquette: 1 frequent combinations
Creighton: 1 frequent combinations
Colorado_St_: 1 frequent combinations
UCLA: 1 frequent combinations
Michigan: 1 frequent combinations
Oregon: 1 frequent combinations
25


**Round of 16 itemsets:**

In [13]:
min_support = .2
frequent_1_itemsets16m = find_frequent_1_itemsets(transactions_made_16, min_support)

candidates16m_2 = generate_candidates(frequent_1_itemsets16m, k=2)

frequent_2_itemsets16m = count_support(candidates16m_2, transactions_made_16, min_support=0.17)

for itemset, support in frequent_2_itemsets16m.items():
    print(itemset, support)

frozenset({'Very_Good_Defense', 'Very_Good_ORB'}) 0.18359375
frozenset({'Very_Good_Defense', 'Very_Good_Team'}) 0.1796875
frozenset({'Very_Good_3P', 'Very_Good_EFG'}) 0.17578125
frozenset({'Very_Good_Offense', 'Very_Good_EFG'}) 0.17578125
frozenset({'Very_Good_Defense', 'Very_Good_EFG_D'}) 0.2109375


In [21]:
vgood_team_and_vgood_def = currdf[
    (currdf['Defense_Category'] == 'Very_Good_Defense') &
    (currdf['Team_Strength_Category'] == 'Very_Good_Team')
]

vgood_efg_and_vgood_off = currdf[
    (currdf['Offense_Category'] == 'Very_Good_Offense') &
    (currdf['Shooting_Efficiency_Category'] == 'Very_Good_EFG')
]

vgood_def_and_vgood_orb = currdf[
    (currdf['Rebounding_Category'] == 'Very_Good_ORB') &
    (currdf['Defense_Category'] == 'Very_Good_Defense')
]

vgood_efg_and_vgood_3p = currdf[
    (currdf['ThreePoint_Category'] == 'Very_Good_3P') &
    (currdf['Shooting_Efficiency_Category'] == 'Very_Good_EFG')
]

vgood_def_and_vgood_efgd = currdf[
    (currdf['Defense_Efficiency_Category'] == 'Very_Good_EFGD') &
    (currdf['Defense_Category'] == 'Very_Good_Defense')
]

teams_vgood_team_and_vgood_def = set(vgood_team_and_vgood_def['Team'])
teams_vgood_efg_and_vgood_off = set(vgood_efg_and_vgood_off['Team'])
teams_vgood_def_and_vgood_orb = set(vgood_def_and_vgood_orb['Team'])
teams_vgood_efg_and_vgood_3p = set(vgood_efg_and_vgood_3p['Team'])
teams_vgood_def_and_vgood_efgd = set(vgood_def_and_vgood_efgd['Team'])

combo_count16 = defaultdict(int)

#Add +1 for every bad trait set a team appears in
for team in teams_vgood_team_and_vgood_def:
    combo_count16[team] += 1
for team in teams_vgood_efg_and_vgood_off:
    combo_count16[team] += 1
for team in teams_vgood_def_and_vgood_orb:
    combo_count16[team] += 1
for team in teams_vgood_efg_and_vgood_3p:
    combo_count16[team] += 1
for team in teams_vgood_def_and_vgood_efgd:
    combo_count16[team] += 1

# Print teams sorted by how many bad combos they match
sorted_teams16 = sorted(combo_count16.items(), key=lambda x: x[1], reverse=True)

print("Teams ranked by number frequent combinations:")
for team, count in sorted_teams16:
    print(f"{team}: {count} frequent combinations")
    
print(len(sorted_teams16))

Teams ranked by number frequent combinations:
Duke: 3 frequent combinations
Houston: 2 frequent combinations
Tennessee: 2 frequent combinations
Purdue: 2 frequent combinations
Florida: 1 frequent combinations
Alabama: 1 frequent combinations
Gonzaga: 1 frequent combinations
BYU: 1 frequent combinations
Auburn: 1 frequent combinations
Missouri: 1 frequent combinations
Utah_St_: 1 frequent combinations
Connecticut: 1 frequent combinations
Texas_Tech: 1 frequent combinations
Kentucky: 1 frequent combinations
Michigan_St_: 1 frequent combinations
St__John_s: 1 frequent combinations
Liberty: 1 frequent combinations
Yale: 1 frequent combinations
18


**Elite 8 Teams:**

In [15]:
min_support = .2
frequent_1_itemsets8m = find_frequent_1_itemsets(transactions_made_8, min_support)

candidates8m_2 = generate_candidates(frequent_1_itemsets8m, k=2)

frequent_2_itemsets8m = count_support(candidates8m_2, transactions_made_8, min_support=0.1)

candidates8m_3 = generate_candidates(frequent_2_itemsets8m, k=3)

frequent_3_itemsets8m = count_support(candidates8m_3, transactions_made_8, min_support=.15)

for itemset, support in frequent_3_itemsets8m.items():
    print(itemset, support)

frozenset({'Very_Good_Defense', 'Very_Good_ORB', 'Very_Good_Team'}) 0.15625
frozenset({'Very_Good_ORB', 'Very_Good_Team', 'Very_Good_Win%'}) 0.15625
frozenset({'Very_Good_Defense', 'Very_Good_EFG_D', 'Very_Good_Team'}) 0.15625
frozenset({'Very_Good_Defense', 'Very_Good_Team', 'Very_Good_Win%'}) 0.15625


In [22]:
vgood_team_and_vgood_def_vgoodefgd = currdf[
    (currdf['Defense_Category'] == 'Very_Good_Defense') &
    (currdf['Team_Strength_Category'] == 'Very_Good_Team') &
    (currdf['Defense_Efficiency_Category'] == 'Very_Good_EFG_D')
]

vgood_team_and_vgood_orb_vgoodwin = currdf[
    (currdf['Rebounding_Category'] == 'Very_Good_ORB') &
    (currdf['Team_Strength_Category'] == 'Very_Good_Team') &
    (currdf['WinRate_Category'] == 'Very_Good_Win%')
]

vgood_team_and_vgood_win_vgood_def = currdf[
    (currdf['WinRate_Category'] == 'Very_Good_Win%') &
    (currdf['Team_Strength_Category'] == 'Very_Good_Team') &
    (currdf['Defense_Category'] == 'Very_Good_Defense')
]

vgood_team_and_vgood_orb_vgood_def = currdf[
    (currdf['Rebounding_Category'] == 'Very_Good_ORB') &
    (currdf['Team_Strength_Category'] == 'Very_Good_Team') &
    (currdf['Defense_Category'] == 'Very_Good_Defense')
]


teams_vgood_team_and_vgood_def_vgoodefgd = set(vgood_team_and_vgood_def_vgoodefgd['Team'])
teams_vgood_team_and_vgood_orb_vgoodwin = set(vgood_team_and_vgood_orb_vgoodwin['Team'])
teams_vgood_team_and_vgood_win_vgood_def = set(vgood_team_and_vgood_win_vgood_def['Team'])
teams_vgood_team_and_vgood_orb_vgood_def = set(vgood_team_and_vgood_orb_vgood_def['Team'])

combo_count8 = defaultdict(int)

#Add +1 for every bad trait set a team appears in
for team in teams_vgood_team_and_vgood_def_vgoodefgd:
    combo_count8[team] += 1
for team in teams_vgood_team_and_vgood_orb_vgoodwin:
    combo_count8[team] += 1
for team in teams_vgood_team_and_vgood_win_vgood_def:
    combo_count8[team] += 1
for team in teams_vgood_team_and_vgood_orb_vgood_def:
    combo_count8[team] += 1


# Step 4: Print teams sorted by how many bad combos they match
sorted_teams8 = sorted(combo_count8.items(), key=lambda x: x[1], reverse=True)

print("Teams ranked by number frequent combinations:")
for team, count in sorted_teams8:
    print(f"{team}: {count} frequent combinations")
    
print(len(sorted_teams8))

Teams ranked by number frequent combinations:
Houston: 4 frequent combinations
Duke: 2 frequent combinations
Tennessee: 2 frequent combinations
Florida: 1 frequent combinations
4


These 4 teams are the likely contenders with Houston being the strongest

**Final 4 itemsets:**

In [17]:
min_support = .2
frequent_1_itemsets4m = find_frequent_1_itemsets(transactions_made_4, min_support)

candidates4m_2 = generate_candidates(frequent_1_itemsets4m, k=2)

frequent_2_itemsets4m = count_support(candidates4m_2, transactions_made_4, min_support=0.1)

candidates4m_3 = generate_candidates(frequent_2_itemsets4m, k=3)

frequent_3_itemsets4m = count_support(candidates4m_3, transactions_made_4, min_support=.15)

candidates4m_4 = generate_candidates(frequent_3_itemsets4m, k=4)

frequent_4_itemsets4m = count_support(candidates4m_4, transactions_made_4, min_support=.15)

for itemset, support in frequent_4_itemsets4m.items():
    print(itemset, support)

frozenset({'Very_Good_ORB', 'Very_Good_Defense', 'Very_Good_Team', 'Very_Good_Win%'}) 0.171875
frozenset({'Very_Good_Offense', 'Very_Good_EFG', 'Very_Good_Team', 'Very_Good_Win%'}) 0.15625
frozenset({'Very_Good_EFG_D', 'Very_Good_ORB', 'Very_Good_Defense', 'Very_Good_Team'}) 0.15625
frozenset({'Very_Good_ORB', 'Very_Good_Offense', 'Very_Good_Team', 'Very_Good_Win%'}) 0.15625
frozenset({'Very_Good_EFG_D', 'Very_Good_Defense', 'Very_Good_Team', 'Very_Good_Win%'}) 0.171875


In [23]:
vgood_team_vgood_def_vgood_orb_vgood_efgd = currdf[
    (currdf['Defense_Category'] == 'Very_Good_Defense') &
    (currdf['Team_Strength_Category'] == 'Very_Good_Team') &
    (currdf['Rebounding_Category'] == 'Very_Good_ORB') &
    (currdf['Defense_Efficiency_Category'] == 'Very_Good_EFG_D')
]
vgood_team_vgood_def_vgood_win_vgood_efgd = currdf[
    (currdf['Defense_Category'] == 'Very_Good_Defense') &
    (currdf['Team_Strength_Category'] == 'Very_Good_Team') &
    (currdf['WinRate_Category'] == 'Very_Good_Win%') &
    (currdf['Defense_Efficiency_Category'] == 'Very_Good_EFG_D')
]
vgood_team_vgood_off_vgood_win_vgood_efg = currdf[
    (currdf['Offense_Category'] == 'Very_Good_Offense') &
    (currdf['Team_Strength_Category'] == 'Very_Good_Team') &
    (currdf['WinRate_Category'] == 'Very_Good_Win%') &
    (currdf['Shooting_Efficiency_Category'] == 'Very_Good_EFG')
]
vgood_team_vgood_off_vgood_orb_vgood_win = currdf[
    (currdf['Offense_Category'] == 'Very_Good_Offense') &
    (currdf['Team_Strength_Category'] == 'Very_Good_Team') &
    (currdf['WinRate_Category'] == 'Very_Good_Win%') &
    (currdf['Rebounding_Category'] == 'Very_Good_ORB')
]
vgood_team_vgood_def_vgood_orb_vgood_win = currdf[
    (currdf['Defense_Category'] == 'Very_Good_Defense') &
    (currdf['Team_Strength_Category'] == 'Very_Good_Team') &
    (currdf['WinRate_Category'] == 'Very_Good_Win%') &
    (currdf['Rebounding_Category'] == 'Very_Good_ORB')
]


teams_vgood_team_vgood_def_vgood_orb_vgood_efgd = set(vgood_team_vgood_def_vgood_orb_vgood_efgd['Team'])
teams_vgood_team_vgood_def_vgood_win_vgood_efgd = set(vgood_team_vgood_def_vgood_win_vgood_efgd['Team'])
teams_vgood_team_vgood_off_vgood_win_vgood_efg = set(vgood_team_vgood_off_vgood_win_vgood_efg['Team'])
teams_vgood_team_vgood_off_vgood_orb_vgood_win = set(vgood_team_vgood_off_vgood_orb_vgood_win['Team'])
teams_vgood_team_vgood_def_vgood_orb_vgood_win = set(vgood_team_vgood_def_vgood_orb_vgood_win['Team'])

combo_count4 = defaultdict(int)

# Add +1 for every bad trait set a team appears in
for team in teams_vgood_team_vgood_def_vgood_orb_vgood_efgd:
    combo_count4[team] += 1
for team in teams_vgood_team_vgood_def_vgood_win_vgood_efgd:
    combo_count4[team] += 1
for team in teams_vgood_team_vgood_off_vgood_win_vgood_efg:
    combo_count4[team] += 1
for team in teams_vgood_team_vgood_off_vgood_orb_vgood_win:
    combo_count4[team] += 1
for team in teams_vgood_team_vgood_def_vgood_orb_vgood_win:
    combo_count4[team] += 1

# Print teams sorted by how many bad combos they match
sorted_teams4 = sorted(combo_count4.items(), key=lambda x: x[1], reverse=True)

print("Teams ranked by number frequent combinations:")
for team, count in sorted_teams4:
    print(f"{team}: {count} frequent combinations")
    
print(len(sorted_teams4))

Teams ranked by number frequent combinations:
Houston: 4 frequent combinations
Duke: 2 frequent combinations
Florida: 2 frequent combinations
Tennessee: 1 frequent combinations
4


These 4 teams are the likely contenders with Houston being the strongest, Duke and Florida being slightly above Tennessee

**Final Itemsets:**

In [19]:
min_support = .2
frequent_1_itemsets2m = find_frequent_1_itemsets(transactions_made_2, min_support)

candidates2m_2 = generate_candidates(frequent_1_itemsets2m, k=2)

frequent_2_itemsets2m = count_support(candidates2m_2, transactions_made_2, min_support=0.1)

candidates2m_3 = generate_candidates(frequent_2_itemsets2m, k=3)

frequent_3_itemsets2m = count_support(candidates2m_3, transactions_made_2, min_support=.15)

candidates2m_4 = generate_candidates(frequent_3_itemsets2m, k=4)

frequent_4_itemsets2m = count_support(candidates2m_4, transactions_made_2, min_support=.15)

candidates2m_5 = generate_candidates(frequent_4_itemsets2m, k=5)

frequent_5_itemsets2m = count_support(candidates2m_5, transactions_made_2, min_support=.15)

candidates2m_6 = generate_candidates(frequent_5_itemsets2m, k=6)

frequent_6_itemsets2m = count_support(candidates2m_6, transactions_made_2, min_support=.15) 

for itemset, support in frequent_6_itemsets2m.items():
    print(itemset, support)

frozenset({'Very_Good_ORB', 'Very_Good_Offense', 'Good_TOR', 'Very_Good_3P', 'Very_Good_Team', 'Very_Good_Win%'}) 0.15625


In [20]:
vgood_team_vgood_off_vgood_orb_vgood_3p_vgood_win_vgood_tor = currdf[
    (currdf['Offense_Category'] == 'Very_Good_Offense') &
    (currdf['Team_Strength_Category'] == 'Very_Good_Team') &
    (currdf['Rebounding_Category'] == 'Very_Good_ORB') &
    (currdf['WinRate_Category'] == 'Very_Good_Win%') &
    (currdf['ThreePoint_Category'] == 'Very_Good_3P') &
    (currdf['Turnover_Category'] == 'Very_Good_TOR')
]

print(vgood_team_vgood_off_vgood_orb_vgood_3p_vgood_win_vgood_tor['Team'])

0    Houston
Name: Team, dtype: object


Based on these itemsets, it feels that Houston is the strongest team in the tournament

Using all these itemsets, we are given a general sense of almost every team and how far they are capable of making it. We now want to comfirm and check with the GMM clustering to see if they match and make a bracket from there. 